# Machine Registry & Dashboard

The machine registry is the address book of the platform. Every other subsystem — the dispatcher, the monitor, the job router — queries it to find available compute. A machine that is not registered does not exist to NBX. This makes the registry the single source of truth for infrastructure state, and it must be kept consistent with reality: machines that go offline must be marked as such, and new machines must be registered before jobs can be routed to them.

## Machine Data Model

The data model follows the standard Pydantic split for CRUD APIs: (1) `MachineCreate` holds the fields the client supplies at registration, (2) `MachineRead` adds server-generated fields (`id`, `registered_at`, `status`) that the client never writes, and (3) `MachineUpdate` marks every field `Optional` so a `PATCH` request can update one field without repeating all the others.

**Tag-based routing.** The `tags` list enables the dispatcher's tag-matching strategy (notebook 04): a job tagged `["gpu", "a100"]` is only sent to machines carrying both tags. Tags are stored as a JSON array column in the database — this avoids a join table at the cost of not being able to index individual tag values efficiently, which is an acceptable trade-off for a registry that is never expected to contain more than a few hundred machines.

**`status` as a derived field.** `MachineStatus` (`online | offline | unknown`) is not set by the client at registration — it defaults to `unknown` and is subsequently written by the health poller as it probes each machine. This separation prevents stale client-supplied status values from shadowing what the poller actually observes.

In [ ]:
from __future__ import annotations

import uuid
from datetime import datetime, timezone
from enum import Enum

from pydantic import BaseModel, Field


class MachineStatus(str, Enum):
    online  = "online"
    offline = "offline"
    unknown = "unknown"


class MachineCreate(BaseModel):
    name:     str
    provider: str               # "ssh", "runpod", "aws"
    host:     str               # IP address or hostname
    tags:     list[str] = []


class MachineUpdate(BaseModel):
    name:   str | None         = None
    host:   str | None         = None
    status: MachineStatus | None = None
    tags:   list[str] | None   = None


class MachineRead(BaseModel):
    id:            str           = Field(default_factory=lambda: uuid.uuid4().hex[:8])
    name:          str
    provider:      str
    host:          str
    status:        MachineStatus = MachineStatus.unknown
    tags:          list[str]     = []
    registered_at: datetime      = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

Validating a registration payload and promoting it to a `MachineRead`:

In [ ]:
payload = MachineCreate(
    name="gpu-01",
    provider="ssh",
    host="192.168.1.100",
    tags=["gpu", "a100"],
)

machine = MachineRead(**payload.model_dump())
print(machine.model_dump_json(indent=2))

## Machine Registry API

The registry exposes five REST endpoints covering the full CRUD surface. An in-memory `dict[str, MachineRead]` serves as the store — the pattern is identical to what a SQLAlchemy `AsyncSession` would expose, so the router code stays unchanged when swapping in a real database.

**Endpoint design.** `POST /machines/` registers a new machine and returns `201 Created`. `GET /machines/` returns all machines, with optional `provider` and `status` query parameters so the dispatcher can fetch only `online ssh` machines without scanning the full registry. `PATCH /machines/{id}` accepts a `MachineUpdate` with only the changed fields — useful for the health poller to write a new `status` without touching the other fields. `DELETE /machines/{id}` unregisters a machine when decommissioning a pod or retiring a server.

The store key is `machine.id` (a UUID string) rather than `machine.name` because names are mutable and not guaranteed unique across providers. Two RunPod pods could both be named `"gpu-worker"` — the UUID ensures no collisions.

Defining the store and router:

In [ ]:
from fastapi import APIRouter, HTTPException, Query

machines: dict[str, MachineRead] = {}

router = APIRouter(prefix="/machines", tags=["machines"])


@router.post("/", response_model=MachineRead, status_code=201)
async def register_machine(payload: MachineCreate) -> MachineRead:
    machine = MachineRead(**payload.model_dump())
    machines[machine.id] = machine
    return machine


@router.get("/", response_model=list[MachineRead])
async def list_machines(
    status: MachineStatus | None = Query(default=None)
) -> list[MachineRead]:
    result = list(machines.values())
    if status is not None:
        result = [m for m in result if m.status == status]
    return result


@router.get("/{machine_id}", response_model=MachineRead)
async def get_machine(machine_id: str) -> MachineRead:
    if machine_id not in machines:
        raise HTTPException(status_code=404, detail="Machine not found")
    return machines[machine_id]


@router.patch("/{machine_id}", response_model=MachineRead)
async def update_machine(machine_id: str, patch: MachineUpdate) -> MachineRead:
    if machine_id not in machines:
        raise HTTPException(status_code=404, detail="Machine not found")
    stored  = machines[machine_id]
    updated = stored.model_copy(
        update={k: v for k, v in patch.model_dump().items() if v is not None}
    )
    machines[machine_id] = updated
    return updated


@router.delete("/{machine_id}", status_code=204)
async def deregister_machine(machine_id: str) -> None:
    if machine_id not in machines:
        raise HTTPException(status_code=404, detail="Machine not found")
    del machines[machine_id]

We mount the router into a minimal FastAPI app and exercise each endpoint with an `httpx` async client:

In [ ]:
import asyncio
import httpx
from fastapi import FastAPI
from httpx import ASGITransport

app = FastAPI()
app.include_router(router)


async def demo():
    machines.clear()
    transport = ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:

        # Register two machines
        r1 = await client.post("/machines/", json={"name": "gpu-01", "provider": "ssh", "host": "10.0.0.1", "tags": ["gpu"]})
        r2 = await client.post("/machines/", json={"name": "cpu-01", "provider": "ssh", "host": "10.0.0.2", "tags": ["cpu"]})
        mid1, mid2 = r1.json()["id"], r2.json()["id"]
        print("Registered:", mid1, mid2)

        # Mark gpu-01 online
        r3 = await client.patch(f"/machines/{mid1}", json={"status": "online"})
        print("Updated status:", r3.json()["status"])

        # List only online machines
        r4 = await client.get("/machines/", params={"status": "online"})
        print("Online:", [m["name"] for m in r4.json()])

        # Delete cpu-01
        await client.delete(f"/machines/{mid2}")
        r5 = await client.get("/machines/")
        print("After delete:", [m["name"] for m in r5.json()])


asyncio.run(demo())

## Provider Adapter Pattern

The `SSHProvider` is the first concrete provider. It reads machines from the registry filtered by `provider == "ssh"`, connects via `asyncssh`, runs diagnostic commands, and parses the output into a normalized metrics dict. The three commands that cover our key metrics are:

- `cat /proc/loadavg` — CPU load averages (1m, 5m, 15m)
- `free -m` — total and used RAM in megabytes
- `df -h /` — disk usage on the root filesystem

We first define `MachineMetrics`:

In [ ]:
from dataclasses import dataclass


@dataclass
class MachineMetrics:
    machine_id:    str
    cpu_load_1m:   float
    ram_total_mb:  float
    ram_used_mb:   float
    disk_total_gb: float
    disk_used_gb:  float

    @property
    def ram_pct(self) -> float:
        return 100.0 * self.ram_used_mb / self.ram_total_mb if self.ram_total_mb else 0.0

    @property
    def disk_pct(self) -> float:
        return 100.0 * self.disk_used_gb / self.disk_total_gb if self.disk_total_gb else 0.0

The full `SSHProvider.get_metrics` implementation — shown as a static code block since it requires `asyncssh`:

```python
# providers/ssh.py — run as a standalone script
import asyncssh


class SSHProvider:
    def __init__(self, key_path: str):
        self.key_path = key_path

    async def get_metrics(self, machine: MachineRead) -> MachineMetrics:
        async with asyncssh.connect(
            machine.host,
            client_keys=[self.key_path],
            known_hosts=None,
        ) as conn:
            load_out = await conn.run("cat /proc/loadavg", check=True)
            mem_out  = await conn.run("free -m",           check=True)
            disk_out = await conn.run("df -h /",           check=True)

        cpu_load = float(load_out.stdout.split()[0])

        # free -m: header line then "Mem: total used free ..."
        mem_line  = mem_out.stdout.splitlines()[1].split()
        ram_total = float(mem_line[1])
        ram_used  = float(mem_line[2])

        # df -h: header line then "/dev/sdX  SIZE USED AVAIL USE% /"
        df_line    = disk_out.stdout.splitlines()[1].split()
        disk_total = float(df_line[1].rstrip("G"))
        disk_used  = float(df_line[2].rstrip("G"))

        return MachineMetrics(
            machine_id=machine.id,
            cpu_load_1m=cpu_load,
            ram_total_mb=ram_total,
            ram_used_mb=ram_used,
            disk_total_gb=disk_total,
            disk_used_gb=disk_used,
        )
```

For completeness, the `RunPodProvider` stub shows how the same interface maps to a REST API:

In [ ]:
import httpx as _httpx


class RunPodProvider:
    """Stub — real implementation calls the RunPod REST API."""

    BASE = "https://api.runpod.io/v2"

    def __init__(self, api_key: str):
        self._headers = {"Authorization": f"Bearer {api_key}"}

    async def list_machines(self) -> list[MachineRead]:
        """GET /pods returns active pod objects; we map each to MachineRead."""
        async with _httpx.AsyncClient() as client:
            r    = await client.get(f"{self.BASE}/pods", headers=self._headers)
            pods = r.json()["data"]["pods"]
        return [
            MachineRead(
                id=pod["id"],
                name=pod["name"],
                provider="runpod",
                host=pod.get("runtime", {}).get("ports", [{}])[0].get("ip", ""),
                status=(
                    MachineStatus.online
                    if pod["desiredStatus"] == "RUNNING"
                    else MachineStatus.offline
                ),
            )
            for pod in pods
        ]

    async def get_metrics(self, machine_id: str) -> MachineMetrics:
        """RunPod has no metrics endpoint — SSH into the pod instead."""
        raise NotImplementedError("Use SSHProvider.get_metrics for RunPod pods.")

:::{.callout-note}
RunPod pods are reachable via SSH once they have a public IP. The typical pattern is to use `RunPodProvider.list_machines()` to discover pod IPs, then use `SSHProvider.get_metrics()` to poll health — mixing providers for different responsibilities is entirely valid given the `BaseProvider` abstraction.

:::

## Async Health Polling

The health poller runs as a background `asyncio` task alongside the FastAPI server. It polls each machine's metrics at a configurable interval and writes results back to the registry. We use `asyncio.gather` to poll all machines concurrently — this matters when there are many machines and each poll takes several hundred milliseconds over SSH.

We define a mock provider for demonstration — it returns random metrics without any network calls:

In [ ]:
import asyncio
import random


class MockProvider:
    """Returns randomized metrics — no real network calls."""

    async def get_metrics(self, machine: MachineRead) -> MachineMetrics:
        await asyncio.sleep(0.05)  # simulate network latency
        return MachineMetrics(
            machine_id=machine.id,
            cpu_load_1m=random.uniform(0.1, 4.0),
            ram_total_mb=32_768.0,
            ram_used_mb=random.uniform(2_000.0, 28_000.0),
            disk_total_gb=500.0,
            disk_used_gb=random.uniform(50.0, 400.0),
        )

The polling loop — updates the in-memory registry and metrics cache on each tick:

In [ ]:
metrics_cache: dict[str, MachineMetrics] = {}


async def poll_metrics(provider: MockProvider, interval: float = 30.0) -> None:
    """Continuously poll all machines and update the metrics cache."""
    while True:
        current = list(machines.values())
        results = await asyncio.gather(
            *[provider.get_metrics(m) for m in current],
            return_exceptions=True,
        )
        for m, result in zip(current, results):
            if isinstance(result, Exception):
                machines[m.id] = machines[m.id].model_copy(
                    update={"status": MachineStatus.offline}
                )
            else:
                metrics_cache[m.id] = result
                machines[m.id] = machines[m.id].model_copy(
                    update={"status": MachineStatus.online}
                )
        await asyncio.sleep(interval)

Running the mock poller for 3 ticks to verify concurrent polling:

In [ ]:
async def run_demo_poll():
    machines.clear()
    metrics_cache.clear()
    for name, host in [("gpu-01", "10.0.0.1"), ("gpu-02", "10.0.0.2")]:
        m = MachineRead(name=name, provider="ssh", host=host, tags=["gpu"])
        machines[m.id] = m

    provider = MockProvider()
    for tick in range(3):
        current = list(machines.values())
        results = await asyncio.gather(*[provider.get_metrics(m) for m in current])
        for m, r in zip(current, results):
            metrics_cache[m.id] = r
            machines[m.id] = machines[m.id].model_copy(
                update={"status": MachineStatus.online}
            )
        print(f"Tick {tick + 1}:")
        for mid, met in metrics_cache.items():
            print(
                f"  {machines[mid].name}: "
                f"CPU={met.cpu_load_1m:.2f}  "
                f"RAM={met.ram_pct:.1f}%  "
                f"Disk={met.disk_pct:.1f}%"
            )
        await asyncio.sleep(0.05)


asyncio.run(run_demo_poll())

:::{.callout-caution}
`asyncio.gather(..., return_exceptions=True)` is essential here. Without it, a single SSH timeout causes the entire gather to raise and the polling loop crashes, leaving all other machines unpolled. With `return_exceptions=True`, failures are returned as `Exception` objects and handled per-machine.

:::

## Flet Live Dashboard

The following is run as a standalone script. Save as `dashboard/main.py` and run with `python dashboard/main.py`.

The dashboard has one tab per provider. Each tab shows a `DataTable` of machines — name, status, CPU load, RAM%, disk% — and three `LineChart` controls tracking each metric over the last 60 samples. A background `asyncio` task polls `GET /machines/` and the metrics endpoint every 30 seconds and pushes new data points into the charts.

```python
# dashboard/main.py — standalone Flet script
from __future__ import annotations

import asyncio
import collections

import flet as ft
import httpx

NBX_BASE      = "http://localhost:8000"
POLL_INTERVAL = 30
HISTORY_LEN   = 60

cpu_history  = collections.defaultdict(lambda: collections.deque(maxlen=HISTORY_LEN))
ram_history  = collections.defaultdict(lambda: collections.deque(maxlen=HISTORY_LEN))
disk_history = collections.defaultdict(lambda: collections.deque(maxlen=HISTORY_LEN))


async def fetch_all_machines() -> list[dict]:
    async with httpx.AsyncClient() as c:
        r = await c.get(f"{NBX_BASE}/machines/")
        return r.json()


async def fetch_metrics(machine_id: str) -> dict:
    async with httpx.AsyncClient() as c:
        r = await c.get(f"{NBX_BASE}/machines/{machine_id}/metrics")
        return r.json()


STATUS_COLORS = {
    "online":  ft.Colors.GREEN_400,
    "offline": ft.Colors.RED_400,
    "unknown": ft.Colors.GREY_500,
}


def status_badge(status: str) -> ft.Container:
    return ft.Container(
        content=ft.Text(status, size=12, color=ft.Colors.WHITE),
        bgcolor=STATUS_COLORS.get(status, ft.Colors.GREY_500),
        border_radius=4,
        padding=ft.Padding(6, 2, 6, 2),
    )


def machine_table(ms: list[dict]) -> ft.DataTable:
    return ft.DataTable(
        columns=[
            ft.DataColumn(ft.Text("Name")),
            ft.DataColumn(ft.Text("Status")),
            ft.DataColumn(ft.Text("CPU load")),
            ft.DataColumn(ft.Text("RAM %")),
            ft.DataColumn(ft.Text("Disk %")),
        ],
        rows=[
            ft.DataRow(cells=[
                ft.DataCell(ft.Text(m["name"])),
                ft.DataCell(status_badge(m["status"])),
                ft.DataCell(ft.Text(
                    f"{cpu_history[m['id']][-1]:.2f}"
                    if cpu_history[m["id"]] else "—"
                )),
                ft.DataCell(ft.Text(
                    f"{ram_history[m['id']][-1]:.1f}%"
                    if ram_history[m["id"]] else "—"
                )),
                ft.DataCell(ft.Text(
                    f"{disk_history[m['id']][-1]:.1f}%"
                    if disk_history[m["id"]] else "—"
                )),
            ])
            for m in ms
        ],
    )


def metric_chart(
    history: collections.deque,
    color: str,
) -> ft.LineChart:
    points = [ft.LineChartDataPoint(x=i, y=v) for i, v in enumerate(history)]
    return ft.LineChart(
        data_series=[ft.LineChartData(data_points=points, stroke_width=2, color=color)],
        left_axis=ft.ChartAxis(labels_size=30),
        bottom_axis=ft.ChartAxis(labels_size=20),
        expand=True,
        height=120,
    )


async def main(page: ft.Page):
    page.title          = "NBX Dashboard"
    page.window.width   = 1100
    page.window.height  = 720

    tabs_ctrl = ft.Tabs(selected_index=0, tabs=[], expand=True)
    page.add(tabs_ctrl)

    async def refresh():
        all_machines = await fetch_all_machines()
        by_provider: dict[str, list[dict]] = {}
        for m in all_machines:
            by_provider.setdefault(m["provider"], []).append(m)

        for m in all_machines:
            try:
                met = await fetch_metrics(m["id"])
                cpu_history[m["id"]].append(met.get("cpu_load_1m", 0))
                ram_history[m["id"]].append(met.get("ram_pct", 0))
                disk_history[m["id"]].append(met.get("disk_pct", 0))
            except Exception:
                pass

        tabs_ctrl.tabs.clear()
        for provider, ms in by_provider.items():
            content = ft.Column(
                controls=[
                    machine_table(ms),
                    ft.Text("CPU Load", weight=ft.FontWeight.BOLD),
                    ft.Row(
                        [metric_chart(cpu_history[m["id"]], ft.Colors.BLUE_400) for m in ms],
                        expand=True,
                    ),
                    ft.Text("RAM %", weight=ft.FontWeight.BOLD),
                    ft.Row(
                        [metric_chart(ram_history[m["id"]], ft.Colors.GREEN_400) for m in ms],
                        expand=True,
                    ),
                    ft.Text("Disk %", weight=ft.FontWeight.BOLD),
                    ft.Row(
                        [metric_chart(disk_history[m["id"]], ft.Colors.ORANGE_400) for m in ms],
                        expand=True,
                    ),
                ],
                scroll=ft.ScrollMode.AUTO,
            )
            tabs_ctrl.tabs.append(ft.Tab(text=provider.upper(), content=content))

        page.update()

    async def poll_loop():
        while True:
            await refresh()
            await asyncio.sleep(POLL_INTERVAL)

    page.run_task(poll_loop)


ft.run(main)
```

**Remark.** The dashboard polls `GET /machines/{id}/metrics` — an endpoint we have not yet defined in the router. In the next notebook we add it alongside the job execution infrastructure. The dashboard is intentionally forward-compatible: it will work as soon as that endpoint exists.

---

■